# franq_ext — real GPU experiments (Colab / Kaggle, free T4)

Runs the full ablation ladder with real models and **real, labelled hallucinations**.

- **LLM:** `Qwen2.5-3B-Instruct` (ungated) or `meta-llama/Llama-3.2-3B-Instruct`.
- **Faithfulness:** a modern **NLI entailment** scorer (torch-2 compatible).
- **Dataset:** **PopQA grouped by subject** (`popqa_structured`). The 3B model answers each attribute from Wikipedia evidence (RAG); wrong answers are the hallucinations we detect/correct.

Runtime → Change runtime type → **GPU** before running. (If you hit the free-GPU daily limit, wait or use Kaggle. The code falls back to CPU but that is far too slow for real runs.)

## 1. Put the project on the path

Upload the WHOLE project folder (the one with `pyproject.toml` and `franq_ext/`). Easiest:
zip it locally, upload the zip, unzip here. This notebook adds the project root to `sys.path`
(no pip install of the package).

In [ ]:
import os, sys, glob

# If you uploaded a zip, unzip it (edit the name to match your file):
# !unzip -o -q /content/franq_ext_project_FRESH.zip -d /content

candidates = glob.glob('/content/**/franq_ext/__init__.py', recursive=True)
assert candidates, 'Could not find franq_ext/. Upload the project folder (with pyproject.toml) to /content.'
PROJECT_ROOT = os.path.dirname(os.path.dirname(candidates[0]))
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print('project root:', PROJECT_ROOT)
import franq_ext; print('franq_ext version:', franq_ext.__version__)
import franq_ext.generation, franq_ext.data.wiki_context
print('structured RAG mode present: OK')

## 2. Install runtime dependencies (torch is already on Colab; do NOT pin/downgrade it)

In [ ]:
!pip install -q -U transformers sentence-transformers faiss-cpu datasets matplotlib
import torch; print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU! Runtime -> Change runtime type -> GPU (or free-GPU limit hit).'

## 3. (Optional) Hugging Face login for gated Llama
Skip if using the ungated Qwen model (the default below).

In [ ]:
# from huggingface_hub import login
# login('hf_XXXX')  # only needed for meta-llama/*

## 4. Configure + run the full ladder

`FRANQ_N` is the number of **entities**. 25 was too few (only ~13 eval, noisy). Use **100+**
for trustworthy numbers, **300** for the final paper run. If you re-run after changing the
Wikipedia-context length, delete the cache first: `!rm -rf .cache`.

In [ ]:
# !rm -rf .cache   # uncomment to refetch fuller Wikipedia contexts
os.environ['FRANQ_MODE'] = 'structured'
os.environ['FRANQ_DATASET'] = 'popqa_structured'
os.environ['FRANQ_N'] = '150'                                # multi-attribute entities to keep
os.environ['FRANQ_MIN_ATTRS'] = '3'                          # KEY: only entities with >=3 attributes (so the graph has real structure)
os.environ['FRANQ_UQ_SAMPLES'] = '3'                         # semantic-entropy samples/fact (5 is slow; 2-3 fine)
os.environ['FRANQ_PROGRESS_EVERY'] = '10'
os.environ['FRANQ_LLM_BACKEND'] = 'hf'
os.environ['FRANQ_LLM_MODEL'] = 'Qwen/Qwen2.5-3B-Instruct'   # or meta-llama/Llama-3.2-3B-Instruct
os.environ['FRANQ_SCORER_BACKEND'] = 'nli'
os.environ['FRANQ_NLI_MODEL'] = 'MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli'
os.environ['FRANQ_RETRIEVER_BACKEND'] = 'dense'
os.environ['FRANQ_DEVICE'] = 'cuda'
os.environ['FRANQ_RESULTS'] = 'results_popqa'

# NOTE: FRANQ_MIN_ATTRS>=3 is what gives Pillar 1 (the graph) something to work on.
# Single-attribute entities make the graph inert. If too few entities qualify, lower to 2.
!python -m franq_ext.experiments.run_all

## 5. Inspect the ablation table and figures

In [ ]:
import pandas as pd
from IPython.display import display, Image
display(pd.read_csv('results_popqa/tables/ablation.csv'))
display(pd.read_csv('results_popqa/tables/regret_analysis.csv'))
for fig in ['fig_auroc.png','fig_prr.png','fig_ece.png','fig_regret.png','fig_factual_accuracy.png']:
    p = f'results_popqa/figures/{fig}'
    if os.path.exists(p):
        display(Image(p))

## Notes

- **Speed:** the 4 conditions share one signal cache (`results_popqa/signal_cache.json`), so
  only the first condition does heavy LLM work; the rest are fast. Semantic-entropy sampling
  (`FRANQ_UQ_SAMPLES`) is the main cost — 2–3 is plenty.
- **Save your GPU budget across sessions:** mount Google Drive and copy the caches there, so
  a second session skips generation entirely:
  ```python
  from google.colab import drive; drive.mount('/content/drive')
  # after a run: !cp -r results_popqa/signal_cache.json .cache /content/drive/MyDrive/franq_cache/
  # next session (before run): !mkdir -p .cache && cp /content/drive/MyDrive/franq_cache/signal_cache.json results_popqa/ ; cp -r /content/drive/MyDrive/franq_cache/.cache/* .cache/ 2>/dev/null || true
  ```
- **Kaggle** gives 30 GPU-hrs/week (vs Colab free's few hrs/day) — a better home for this. Same code; upload the zip as a Kaggle Dataset.
- **What to look for:** ECE dropping B0→A1 (Pillar 2); A2 (+graph) ≥ A1 on AUROC/PRR/ECE (Pillar 1, needs `FRANQ_MIN_ATTRS≥3`); A3 acc_after ≥ acc_before with low regret (Pillar 3).
- **Smaller/faster model** if you're tight on time: set `FRANQ_LLM_MODEL='Qwen/Qwen2.5-1.5B-Instruct'` (2–3× faster than 3B).
- **TriviaQA / NQ** are single-answer (no graph); a secondary detection-only run.